<a href="https://colab.research.google.com/github/sachacn02/Robust_Opt/blob/main/portfolio_problem_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

SIMPLE PORTFOLIO EXAMPLE FROM "The Price of Robustess"

Modified by:
 1. Melvyn   (Created for ROME, 10 Sep 2009)
 2. Erick Delage (Modified 14 April 2015)
 3. Erick Delage (Adapted to RSOME in November 2020)

As discussed in example 1.2 of the  [lecture notes](http://tintin.hec.ca/pages/erick.delage/MATH80624_LectureNotes.pdf) of MATH80624 at HEC Montréal.

WARNING!!!

The following code exploits a free Mosek licence for the course "MATH80624A" offered at HEC Montréal (expiration December 31st 2025). If you have error messages informing you about licencing issues, you may try uncommenting the installation lines for Gurobi. Otherwise, we recommend that you obtain your own licence of either Mosek ([url](https://www.mosek.com/)) or Gurobi ([url](https://www.gurobi.com/)).

# **Preliminaries**

In [ ]:
!pip install rsome
!pip install mosek
!rm mosek.lic
!git clone https://github.com/erickdelage/80624
!cp ./80624/mosek.lic .
#!cp ./80624/msk_solver.py /usr/local/lib/python3.6/dist-packages/rsome #fixes an issue with rsome for quadratic cones
!rm -r ./80624
!mkdir -p /root/mosek
!cp ./mosek.lic /root/mosek
#!pip install -i https://pypi.gurobi.com gurobipy


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.8/87.8 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.4/15.4 MB 86.1 MB/s eta 0:00:00
rm: cannot remove 'mosek.lic': No such file or directory
Cloning into '80624'...
remote: Enumerating objects: 36, done.
remote: Counting objects: 100% (36/36), done.
remote: Compressing objects: 100% (30/30), done.
remote: Total 36 (delta 9), reused 32 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (36/36), 32.62 KiB | 16.31 MiB/s, done.
Resolving deltas: 100% (9/9), done.


In [ ]:
import rsome as rso
import numpy as np
from rsome import ro
from rsome import msk_solver as my_solver  #Import Mosek solver interface
#from rsome import grb_solver as my_solver  #Import Gurobi solver interface


# **Simple Portfolio Example**

In [ ]:
# Parameter setup
n = 150                                     # Number of stocks
i = np.arange(1, n+1)                       # Indices of stocks
mu =0.15 + i*0.05/150                       # Mean returns
sigma = 0.05/450 * (2*i*n*(n+1))**0.5       # Standard deviations of returns
Gamma = 4                                   # Maximum number of estimates that might deviate


## **Solving a deterministic model**

We consider a portfolio construction problem consisting of $n$ stocks. We consider that stock $i$ has future return $r_i$ and are looking for the portfolio composition that will maximize total return on our investment. This can be done by solving the following linear program:
\begin{align}
\max\limits_{x \in \mathbb{R}^n} \;\;&\sum\limits_{i=1}^n r_ix_i \\
\text{subject to}\; & \sum\limits_{i=1}^n x_i =100\% \\
                    & x_i \geq 0, \forall i=1,\cdots,n,
\end{align}
where $x_i$ is the proportion of the budget invested in stock $i$, the first constraint implies that we wish that all the budget be invested, and where $x_i\geq0$ implies that we wish to avoid short selling a stock.

In [ ]:
#Create portfolio model
model = ro.Model('portfolio')

#Portfolio weights
x = model.dvar(n)                           # Fractions of investment

# Objective to maximize the return
model.max(mu@x.T)

# Constraint to invest all the wealth available
model.st(x.sum() == 1)                      # Summation of x is one
# Constraint that weights are positive
model.st(x >= 0)                            # x is non-negative

model.solve(my_solver)

optobj_det = model.get() #get optimal objective value
xx_det   = x.get() #get optimal portfolio

print('Deterministic solution: Expected performance of ',np.round(100*optobj_det,2),'% in return')


Being solved by Mosek...
Solution status: Optimal
Running time: 0.0371s
Deterministic solution: Expected performance of  20.0 % in return


## **Evaluating the worst-case return**

In reality, we would need to consider that there is uncertainty about the return obtained from stock $i$. We call this uncertain return $\tilde{r}_i$ and characterize it using
$$\tilde{r}_i = \mu_i+\sigma_i z_i,$$
where $\mu_i$ is the expected return, and $\sigma_i$ describes the volatility of the return, finally $z_i$ captures the source of the uncertainty about $\tilde{r}_i$. You may for example, think of each $z_i$ as being independently distributed according to a standard normal distribution. We will consider the following, problem instance with $n = 150$:
$$\mu_i = 0.15+ i\frac{0.05}{150},\;\;\; \sigma_i = \frac{0.05}{450}\sqrt{2in(n+1)},\;\;\; z_i \in [-1, 1].$$

We can also quantify our tolerance toward risk by assuming that at most $\Gamma$ number of stocks will have their return diverge from their expected value. By controlling the value of $\Gamma$, one can capture the idea that he is more or less risk averse.

Next, given the optimal proportion of deterministic model (denoted by $x^{\text{det}}_i$), we could attempt to optimize the worst-case return vector by solving the following problem:
\begin{align}
\min\limits_{z \in \mathbb{R}}\;\;&\sum\limits_{i=1}^n ( \mu_i+\sigma_i z_i)x^{\text{det}}_i \\
\text{subject to}\; & |z_i| \leq 1 && \forall i=1,\cdots,n \\
                    & \sum\limits_{i=1}^{n}|z_i| \leq \Gamma.
                    %& z_i \in \mathbb{R} &&\forall i=1,\cdots,n.
\end{align}

In [ ]:
model = ro.Model('worstcaseReturn')

#Optimize the worst-case return vector
z = model.dvar(n) # Portfolo returns
model.min((mu+sigma*z)@xx_det); # Objective to minimize the expected return
model.st(z<=1); #each parameter is between [-1, 1]
model.st(z>=-1)
model.st(rso.norm(z,1)<=Gamma); # sum of absolute deviation smaller than Gamma

model.solve(my_solver); # solve the model
wc_det = model.get();
wc_z_det   = z.get();

print('Deterministic solution: Worst-case performance of ',np.round(100*wc_det,2),'% in return')


Being solved by Mosek...
Solution status: Optimal
Running time: 0.0145s
Deterministic solution: Worst-case performance of  -8.96 % in return


## **Solving a robust model**

Alternatively, we could attempt to solve the following robust optimization problem that only worries about the worst-case return:
\begin{align}
\max\limits_{x \in \mathbb{R}^n} \min\limits_{z \in \mathcal{Z}}\;\;&\sum\limits_{i=1}^n ( \mu_i+\sigma_i z_i)x_i \\
\text{subject to}\; & \sum\limits_{i=1}^n x_i =100\% \\
                    & x_i \geq 0, \forall i=1,\cdots,n,
\end{align}
where the budget uncertainty set $\mathcal{Z}:=\{z \in \mathbb{R}\;|\;|z_i| \leq 1, \sum_{i}|z_i| \leq \Gamma\}$.

In [ ]:
# create a model
model = ro.Model('robustPortfolio')

# Declare uncertain parameters
z = model.rvar(n)  # Random variables
budgetSet = (z<=1, z>=-1,  #each parameter is between [-1, 1]
                  rso.norm(z,1)<=Gamma);   # Budget of uncertainty approach

# Portfolio weights
x=model.dvar(n);

# Objective to maximize the return
model.maxmin((mu+sigma*z)@x,budgetSet);
# Note that r is uncertain and depends in z.
# The objective should be interpreted as
# max{y : y<= (mu + sigma.*z)'x for z in uncertainty set}

# Constraint to invest all the wealth available
model.st(x.sum()==1);
# Constraint that weights are positive
model.st(x>=0);

# solve the model
model.solve(my_solver);
optobj_rob = model.get() #get optimal objective value
xx_rob   = x.get() #get optimal portfolio

print('Robust solution: Expected performance of ',np.round(mu@xx_rob*100,2),'% in return');

print('Robust solution: Worst-case performance of ',np.round(optobj_rob*100,2),'% in return')

Being solved by Mosek...
Solution status: Optimal
Running time: 0.0155s
Robust solution: Expected performance of  18.62 % in return
Robust solution: Worst-case performance of  17.38 % in return
